# Tema 2
## Ex 1
Probabilitatea conditionata se calculează cu formula

$$P(A|B)= \frac{P(A \cap B)}{P(B)}$$

Generam toate cele 36 de rezultate posibile (X, Y) si definim o functie care calculeaza probabilitatea unui eveniment (cazuri favorabile/cazuri posibile).

In [12]:
from itertools import product
from fractions import Fraction

# toate perechile (X, Y), cu X, Y de la 1-6
spatiu= list(product(range(1, 7), repeat=2))
print("Nr rezultate posibile:", len(spatiu))

def P(eveniment):
    # cazuri fav / cazuri pos
    favorabile = sum(1 for (x, y) in spatiu if eveniment(x, y))
    return Fraction(favorabile, len(spatiu))

def P_cond(A, B):
    # P(A|B) = P(A și B) / P(B)
    return P(lambda x, y: A(x, y) and B(x, y)) / P(B)

Nr rezultate posibile: 36


Definim evenimentele:
- A: suma ≥ 10
- B: primul zar este par
- C: cel putin un zar este par
- D: ambele zaruri sunt pare
- E: suma este 8

In [13]:
A= lambda x, y: x + y >= 10
B = lambda x, y: x % 2 == 0
C= lambda x, y: x % 2 == 0 or y % 2 == 0
D = lambda x, y: x % 2 == 0 and y % 2 == 0
E = lambda x, y: x + y == 8

### 1) P(suma ≥ 10 | primul zar par)

In [14]:
r1 = P_cond(A, B)
print("P(A|B) =", r1, "≈", float(r1))

P(A|B) = 2/9 ≈ 0.2222222222222222


Primul zar par → 18 cazuri. Dintre ele, suma≥10 apare în 4 cazuri: (4,6),(6,4), (6,5),(6,6). Deci 4/18 = 2/9.

### 2) P(suma≥10 | cel puțin un zar par)

In [15]:
r2 = P_cond(A, C)
print("P(A|C) =", r2, "≈", float(r2))

P(A|C) = 5/27 ≈ 0.18518518518518517


Cel puțin un zar par → 27 de cazuri (36 minus cele 9 cu ambele impare). Din ele, suma≥10 apare în 5 cazuri: (4,6), (6,4), (5,6),(6,5), (6,6). Nu numaram (5,5), pt cqa ambele zaruri sunt impare. Deci 5/27.

### 3) P(ambele zaruri pare | suma = 8)

In [16]:
r3 = P_cond(D, E)
print("P(D|E) =", r3, "=", float(r3))

P(D|E) = 3/5 = 0.6


Suma 8 se obtine în 5 moduri:(2,6), (3,5), (4,4), (5,3), (6,2). Din ele 3 au ambele zaruri pare:(2,6), (4,4),(6,2).Rezulta 3/5.

### 4) Sunt A și B independente?
Independente daca $P(A \cap B) = P(A) \cdot P(B)$.

In [17]:
P_A = P(A)
P_B = P(B)
P_AB= P(lambda x, y: A(x, y) and B(x, y))

print("P(A)=", P_A)
print("P(B)=", P_B)
print("P(A)*P(B)=", P_A * P_B)
print("P(A și B) =", P_AB)
print("Independente?", P_AB == P_A * P_B)

P(A)= 1/6
P(B)= 1/2
P(A)*P(B)= 1/12
P(A și B) = 1/9
Independente? False


**Concluzie:** 1/9 ≠ 1/12, deci A și B nu sunt independente. primul zar e par face suma mare mai probabila.

### simulare cu 100.000 de aruncări
Aruncam zarurile pe calculator si estimam probabilitatea cond. ca:

$$\text{proportia cazurilor cu A, dintre cele in care a avut loc B}$$

In [18]:
import numpy as np

rng = np.random.default_rng(42)
N = 100_000
X = rng.integers(1, 7, N)# zar 1
Y = rng.integers(1, 7, N)# zar 2

# evenimentele ca vectori True/False
sA = (X + Y) >= 10
sB = (X % 2 == 0)
sC = (X % 2 == 0) | (Y % 2 == 0)
sD = (X % 2 == 0) & (Y % 2 == 0)
sE = (X + Y) == 8

# media dintre cazurile în care conditia e adevărata
est1 = sA[sB].mean()
est2 = sA[sC].mean()
est3 = sD[sE].mean()

print(f"1) simulat: {est1:.4f} , exact: {float(r1):.4f}")
print(f"2) simulat: {est2:.4f} , exact: {float(r2):.4f}")
print(f"3) simulat: {est3:.4f} , exact: {float(r3):.4f}")

# independenta: P(A și B) vs P(A)*P(B)
print(f"4) P(A și B) simulat: {(sA & sB).mean():.4f} , P(A)*P(B) simulat: {sA.mean() * sB.mean():.4f}")

1) simulat: 0.2241 , exact: 0.2222
2) simulat: 0.1854 , exact: 0.1852
3) simulat: 0.5965 , exact: 0.6000
4) P(A și B) simulat: 0.1119 , P(A)*P(B) simulat: 0.0832


**Comparatie:** valorile simulate sunt foarte apropiate de cele exacte (difera doar la a doua-a treia zecimala). Cu cat facem mai multe aruncari, cu atat estimarea se apropie mai mult de valoarea exacta.

---
## Ex 2
**Formula lui Bayes:**

$$P(\text{spam}|\text{cuvânt}) = \frac{P(\text{cuvânt}|\text{spam}) \cdot P(\text{spam})}{P(\text{cuvânt})}$$

Numitorul se calculeaza cu formula probabilitatii totale:

$$P(\text{cuvant}) = P(\text{cuvant}|\text{spam})P(\text{spam}) + P(\text{cuvant}|\text{legitim})P(\text{legitim})$$

In [19]:
# datele problemei
P_spam = 0.30
P_legit = 1-P_spam # 0.70

P_oferta_spam= 0.80
P_oferta_legit = 0.15

P_gratuit_spam= 0.60
P_gratuit_legit = 0.10

### 1) P(spam | "oferta")

In [20]:
# probabilitatea totală a cuv"oferta"
P_oferta = P_oferta_spam * P_spam + P_oferta_legit * P_legit

# Bayes
P_spam_oferta = P_oferta_spam * P_spam / P_oferta
print(f"P(\"oferta\") = {P_oferta:.4f}")
print(f"P(spam | oferta) = {P_spam_oferta:.4f}")

P("oferta") = 0.3450
P(spam | oferta) = 0.6957


**Explicație:** 0,24 / 0,345 ≈ **0,696**, deci ~69,6% din mesajele cu "oferta" sunt spam.

### 2) P(spam | "oferta" și "gratuit")
Cu independenta conditionala (in interiorul fiecarei clase), probabilitatile se inmultesc:
$P(\text{ambele}|\text{spam}) = P(\text{oferta}|\text{spam}) \cdot P(\text{gratuit}|\text{spam})$ ,la fel pentru legitim.

In [21]:
# P(ambele | spam) și P(ambele | legitim)
P_ambele_spam= P_oferta_spam * P_gratuit_spam
P_ambele_legit = P_oferta_legit * P_gratuit_legit

# probabilitatea totala a celor două cuv împreuna
P_ambele = P_ambele_spam * P_spam + P_ambele_legit * P_legit

# Bayes
P_spam_ambele= P_ambele_spam * P_spam / P_ambele
print(f"P(spam | oferta si gratuit) = {P_spam_ambele:.4f}")

P(spam | oferta si gratuit) = 0.9320


**Explicatie:** 0,144 / 0,1545 ≈ **0,932**. Al doilea cuv creste mult sansa ca mesajul să fie spam (de la ~70% la ~93%).

### 3) De ce poate fi nerealista independenta conditionata?
Presupunem ca, dacă stim dacă mesajul e spam sau nu, cuvintele apar independent unul de altul. In realitate:
- cuvintele apar adesea impreuna (de ex. "oferta" și "gratuit" in aceleasi reclame);
- aparitia unui cuvant depinde de context si de stilul mesajului, nu doar de clasa;
- dacă ignoram aceste legaturi, supraestimam certitudinea (probabilitatea ajunge prea aproape de 1), pentru că numaram de doua ori aceeasi informatie.

### 4) Regula de clasificare
Calculam P(spam | cuvintele din mesaj) si comparam cu un prag (de obicei 0,5):
- daca $P(\text{spam}|\text{mesaj})>0{,}5$ → spam
- altfel → legitim

Pragul se poate mari daca vrem să evitam să marcam greșit mesaje legitime drept spam.

In [22]:
def clasifica(are_oferta, are_gratuit, prag=0.5):
    # probabilitatea mesajului daca e spam / daca e legitim
    # (dacă cuv lipseste, folosim 1-probabilitatea)
    p_s= P_spam
    p_l= P_legit
    p_s *= P_oferta_spam if are_oferta else 1-P_oferta_spam
    p_l *= P_oferta_legit if are_oferta else 1-P_oferta_legit
    p_s *= P_gratuit_spam if are_gratuit else 1 - P_gratuit_spam
    p_l *= P_gratuit_legit if are_gratuit else 1 - P_gratuit_legit

    p_spam=p_s / (p_s + p_l)
    return p_spam, "SPAM" if p_spam > prag else "LEGITIM"

for oferta in (True, False):
    for gratuit in (True, False):
        p, eticheta= clasifica(oferta, gratuit)
        print(f"oferta={oferta!s:5} gratuit={gratuit!s:5}->P(spam)={p:.3f}->{eticheta}")

oferta=True  gratuit=True ->P(spam)=0.932->SPAM
oferta=True  gratuit=False->P(spam)=0.504->SPAM
oferta=False gratuit=True ->P(spam)=0.377->LEGITIM
oferta=False gratuit=False->P(spam)=0.043->LEGITIM
